In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

In [ ]:
import importlib
import package_files.benefits_defns
importlib.reload(package_files.benefits_defns)


In [ ]:
usdf_salary = pd.read_parquet('../data/us_10m_nointernship_2018_2024_benefits.parquet.gzip')

In [ ]:
usdf_salary.columns

In [ ]:
usdf_salary = usdf_salary[usdf_salary['SALARY'].notnull()]

In [ ]:
usdf_salary.columns

In [ ]:
def median_ci(data, confidence=0.95):
    data = np.array(data)
    n = len(data)
    median = np.median(data)
    z = stats.norm.ppf(0.5 + confidence / 2.0)
    margin_of_error = z * np.std(data, ddof=1) / np.sqrt(n)
    return median, median - margin_of_error, median + margin_of_error

In [ ]:
for benefit in benefits4:
    grouped_salary = usdf_salary.groupby(['YEAR', 'AI ROLE', benefit])['SALARY'].apply(
        lambda x: pd.Series(median_ci(x, confidence=0.95), index=['Median', 'CI_Lower', 'CI_Upper'])
    ).reset_index()
    # Convert the 'AI ROLE' and benefit to categorical if they are not already
    grouped_salary['AI ROLE'] = grouped_salary['AI ROLE'].astype(str)
    grouped_salary['SALARY'] = pd.to_numeric(grouped_salary['SALARY'], errors='coerce')
    grouped_salary['YEAR'] = grouped_salary['YEAR'].astype(int)

    grouped_salary[benefit] = grouped_salary[benefit].astype(str)

    median_data = grouped_salary[grouped_salary['level_3'] == 'Median']

    # Plotting
    plt.figure(figsize=(10, 6))

    # Use Seaborn's lineplot to plot the median salaries
    sns.lineplot(
        data=median_data,
        x='YEAR',
        y='SALARY',
        hue='AI ROLE',
        style=benefit,
        markers=False,
        errorbar=None
    )

    # Adding error bars manually
    for skill in median_data['AI ROLE'].unique():
        for remote in grouped_salary[benefit].unique():
            subset = grouped_salary[(grouped_salary['AI ROLE'] == skill) & (grouped_salary[benefit] == remote)]
            medians = subset[subset['level_3'] == 'Median']
            ci_lowers = subset[subset['level_3'] == 'CI_Lower']['SALARY'].astype(float).values
            ci_uppers = subset[subset['level_3'] == 'CI_Upper']['SALARY'].astype(float).values
            yerr = np.array([medians['SALARY'].values - ci_lowers, ci_uppers - medians['SALARY'].values])

            plt.errorbar(
                x=medians['YEAR'],
                y=medians['SALARY'],
                yerr=yerr,
                fmt='o',
                color='black'
            )

    plt.xticks(median_data['YEAR'].unique())
    # Customize the legend
    handles, labels = plt.gca().get_legend_handles_labels()
    print(handles, labels)
    # Extract the relevant handles and labels
    new_handles = handles
    new_labels = ['AI Skills','No', 'Yes', f'{benefits_labels_map[benefit]}','No', 'Yes']
    plt.legend(handles=new_handles, labels=new_labels, loc = 'upper left')
    # set legend location
    # plt.legend(['','No AI Skills','AI Skills','','Not Remote','Remote'])
    # plt.legend(handles=handles[:4], labels=new_labels, title='Job Type and Remote Work')
    plt.xlabel(None)
    plt.ylabel('Median Annual Salary (USD)')
    plt.title(benefits_labels_map[benefit])
    # plt.title('Median Annual Salary by AI Skills and Remote Work with 95% Confidence Intervals')
    
    # if salary folder doesn't exist, create it
    if not os.path.exists('../figures/salary'):
        os.makedirs('../figures/salary')
    plt.savefig(f'../figures/salary/salary_by_{benefit}.png')

    plt.show()

# Updated Colors

In [ ]:
for benefit in benefits4:
    grouped_salary = usdf_salary.groupby(['YEAR', 'AI ROLE', benefit])['SALARY'].apply(
        lambda x: pd.Series(median_ci(x, confidence=0.95), index=['Median', 'CI_Lower', 'CI_Upper'])
    ).reset_index()
    # Convert the 'AI ROLE' and benefit to categorical if they are not already
    grouped_salary['AI ROLE'] = grouped_salary['AI ROLE'].astype(str)
    grouped_salary['SALARY'] = pd.to_numeric(grouped_salary['SALARY'], errors='coerce')
    grouped_salary['YEAR'] = grouped_salary['YEAR'].astype(int)

    grouped_salary[benefit] = grouped_salary[benefit].astype(str)
    print(grouped_salary.head())

    median_data = grouped_salary[grouped_salary['level_3'] == 'Median']
    print(median_data.head())
    palette = [benefit_colors[benefit], 'gray']

    # Plotting
    plt.figure(figsize=(10, 6))

    # Use Seaborn's lineplot to plot the median salaries
    sns.lineplot(
        data=median_data,
        x='YEAR',
        y='SALARY',
        hue='AI ROLE',
        style=benefit,
        markers=False,
        errorbar=None, 
        palette=palette, 
        style_order=['True', 'False'], 
        hue_order=['True', 'False']
    )

    # Adding error bars manually
    for skill in median_data['AI ROLE'].unique():
        for remote in grouped_salary[benefit].unique():
            subset = grouped_salary[(grouped_salary['AI ROLE'] == skill) & (grouped_salary[benefit] == remote)]
            medians = subset[subset['level_3'] == 'Median']
            ci_lowers = subset[subset['level_3'] == 'CI_Lower']['SALARY'].astype(float).values
            ci_uppers = subset[subset['level_3'] == 'CI_Upper']['SALARY'].astype(float).values
            yerr = np.array([medians['SALARY'].values - ci_lowers, ci_uppers - medians['SALARY'].values])

            plt.errorbar(
                x=medians['YEAR'],
                y=medians['SALARY'],
                yerr=yerr,
                fmt='o',
                color='black'
            )

    plt.xticks(median_data['YEAR'].unique())
    # Customize the legend
    handles, labels = plt.gca().get_legend_handles_labels()
    print(handles, labels)
    # Extract the relevant handles and labels
    new_handles = handles
    new_labels = ['AI Role','Yes', 'No', f'{benefits_labels_map[benefit]}','Yes', 'No']
    if benefit == benefits4[0]:
        plt.legend(handles=new_handles, labels=new_labels, loc = 'upper left')
    else:
        plt.legend().remove()
    # set legend location
    # plt.legend(['','No AI Skills','AI Skills','','Not Remote','Remote'])
    # plt.legend(handles=handles[:4], labels=new_labels, title='Job Type and Remote Work')
    plt.xlabel(None)
    plt.ylabel('Median Annual Salary (USD)')
    plt.title(benefits_labels_map[benefit])
    # plt.title('Median Annual Salary by AI Skills and Remote Work with 95% Confidence Intervals')
    
    # if salary folder doesn't exist, create it
    if not os.path.exists('../figures/salary'):
        os.makedirs('../figures/salary')
    plt.savefig(f'../figures/salary/salary_by_{benefit}_color.png')

    plt.show()

In [ ]:
usdf_salary['WLB'].dtype

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

# Define the number of rows and columns for subplots
nrows = 2
ncols = 3

# Set up the overall figure
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 12))
axes = axes.flatten()  # Flatten to easily index axes

for i, benefit in enumerate(benefits4):
    grouped_salary = usdf_salary.groupby(['YEAR', 'AI ROLE', benefit])['SALARY'].apply(
        lambda x: pd.Series(median_ci(x, confidence=0.95), index=['Median', 'CI_Lower', 'CI_Upper'])
    ).reset_index()

    # Convert columns to appropriate data types
    grouped_salary['AI ROLE'] = grouped_salary['AI ROLE'].astype(str)
    grouped_salary['SALARY'] = pd.to_numeric(grouped_salary['SALARY'], errors='coerce')
    grouped_salary['YEAR'] = grouped_salary['YEAR'].astype(int)
    grouped_salary[benefit] = grouped_salary[benefit].astype(str)

    median_data = grouped_salary[grouped_salary['level_3'] == 'Median']
    palette = [benefit_colors[benefit], 'gray']

    # Select the current subplot axis
    ax = axes[i]

    # Plot the median salaries using Seaborn
    sns.lineplot(
        data=median_data,
        x='YEAR',
        y='SALARY',
        hue='AI ROLE',
        style=benefit,
        markers=False,
        errorbar=None,
        palette=palette,
        style_order=['True', 'False'],
        hue_order=['True', 'False'],
        ax=ax
    )

    # Adding error bars manually
    for skill in median_data['AI ROLE'].unique():
        for remote in grouped_salary[benefit].unique():
            subset = grouped_salary[(grouped_salary['AI ROLE'] == skill) & (grouped_salary[benefit] == remote)]
            medians = subset[subset['level_3'] == 'Median']
            ci_lowers = subset[subset['level_3'] == 'CI_Lower']['SALARY'].astype(float).values
            ci_uppers = subset[subset['level_3'] == 'CI_Upper']['SALARY'].astype(float).values
            yerr = np.array([medians['SALARY'].values - ci_lowers, ci_uppers - medians['SALARY'].values])

            ax.errorbar(
                x=medians['YEAR'],
                y=medians['SALARY'],
                yerr=yerr,
                fmt='o',
                color='black'
            )

    # Customize ticks and labels for each subplot
    ax.set_xticks(median_data['YEAR'].unique())
    ax.set_xlabel(None)
    ax.set_ylabel('Median Annual Salary (USD)')
    ax.set_title(benefits_labels_map[benefit])

    # Customize legend for each subplot
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
        new_labels = ['AI Role', 'Yes', 'No', f'{benefits_labels_map[benefit]}', 'Yes', 'No']
        ax.legend(handles=handles, labels=new_labels, loc='upper left')
    else:
        ax.legend().remove()

# Adjust layout and save the figure
plt.tight_layout()

# Ensure the directory for saving exists
if not os.path.exists('../figures/salary'):
    os.makedirs('../figures/salary')
plt.savefig('../figures/salary/salary_by_benefit_combined.png')

plt.show()
